# 01 · Authoring fundamentals

**Learning goals**

1. Understand `TaskEnvironment` — the unit of image/resource/config grouping in v2
2. Write sync and async tasks; know when each is appropriate
3. Build container images declaratively (and where they land on GCP)
4. Request CPU/memory/GPU/disk/shm resources, with request/limit ranges
5. Compose multi-environment pipelines and override configuration at call time

If you're coming from Flyte/Union **v1**: there is no `@workflow` decorator in v2 —
composition is plain Python inside tasks. The full mapping is in
[08-v1-to-v2-migration](./08-v1-to-v2-migration.ipynb).

In [ ]:
import flyte

flyte.init_from_config()

## 1. TaskEnvironment anatomy

A `TaskEnvironment` groups tasks that share an image, resources, and execution settings.
One environment ↔ one pod spec, roughly. The important constructor arguments:

| Argument | What it controls |
|---|---|
| `name` | Prefix for pod names and UI grouping |
| `image` | Container image (built for you — next section) |
| `resources` | CPU / memory / GPU / disk / shm per task pod |
| `env_vars` | Plain environment variables |
| `secrets` | Injected secrets (covered in [05](./05-gcp-data-and-integrations.ipynb)) |
| `depends_on` | Other environments this one references (deploy ordering + image reuse) |
| `cache` | Default cache policy for all tasks in the env ([03](./03-caching-performance-reproducibility.ipynb)) |
| `reusable` | Warm-container pool ([04](./04-reusable-containers.ipynb)) |
| `interruptible` / `queue` | Spot instances and node-pool targeting ([03](./03-caching-performance-reproducibility.ipynb)) |
| `plugin_config` | Hands the pod over to a compute plugin, e.g. Ray ([06](./06-ray-on-union.ipynb)) |

`retries` and `timeout` are **per-task** (on `@env.task`), not per-environment.

In [ ]:
env = flyte.TaskEnvironment(
    name="fundamentals",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    env_vars={"PIPELINE_STAGE": "workshop"},
)

## 2. Sync vs async tasks

Both work. Async is the **primary** model in v2 because it makes fan-out natural
(`asyncio.gather`, `flyte.map.aio`) and is required for concurrent execution inside
reusable containers. Rule of thumb:

- Task **calls other tasks** or does I/O-bound work → `async def`
- Leaf task doing pure CPU work → `def` is fine

In [ ]:
import os


@env.task
def normalize(text: str) -> str:          # sync leaf task
    return text.strip().lower()


@env.task
async def pipeline(text: str) -> str:     # async orchestrating task
    cleaned = await normalize.aio(text)   # call sync tasks from async with .aio
    stage = os.environ["PIPELINE_STAGE"]  # env_vars from the TaskEnvironment
    return f"[{stage}] {cleaned}"

### Local vs remote execution

- **Plain Python call** — `pipeline("HeLLo")` runs the function body in your kernel.
  No cluster, no tracking. Great for unit tests.
- **`flyte.run`** — ships the code to the cluster; every task call becomes a tracked,
  isolated container execution with its own resources and logs.

In [ ]:
run = await flyte.run.aio(pipeline, text="  HeLLo GCP  ")
print(run.url)
await run.wait.aio()
run.outputs()

## 3. Container images

`flyte.Image` is a declarative, layered image builder — think `ImageSpec` (v1), rebuilt.
Start from `from_debian_base()` (ships with `flyte` pre-installed) and chain layers.
Images are **content-addressed**: identical definitions reuse the cached build.

On your deployment, builds run in-cluster and land in **Artifact Registry**
(`GAR_REGISTRY` in `.env`), which the dataplane can already pull from.

In [ ]:
ml_image = (
    flyte.Image.from_debian_base(name="workshop-ml", python_version=(3, 12))
    .with_apt_packages("libgomp1")
    .with_pip_packages("scikit-learn==1.5.2", "pandas==2.2.3", "pyarrow")
    .with_env_vars({"OMP_NUM_THREADS": "2"})
)

# For images in a private registry that needs credentials, add a registry secret:
# private_image = flyte.Image.from_debian_base(
#     name="internal-tools",
#     registry="us-central1-docker.pkg.dev/my-project/private-repo",
#     registry_secret="gar-pull-secret",     # created with: flyte create secret ...
# )

# Only for non-Debian bases (e.g. CUDA) — flyte is NOT pre-installed, add it yourself:
# cuda_image = (
#     flyte.Image.from_base("nvidia/cuda:12.4.1-runtime-ubuntu22.04")
#     .with_pip_packages("flyte==2.5.7", "torch")
# )

## 4. Resources

`flyte.Resources` covers CPU, memory, GPU, ephemeral disk, and shared memory. Two forms:

- Single value → request == limit: `cpu="2"`
- Tuple → (request, limit): `cpu=("1", "4")` — the pod can burst

For GPUs, always name the device type — `gpu="1"` schedules on *any* GPU node, which on a
multi-pool cluster makes scheduling non-deterministic and can waste an A100 on a T4 job.

In [ ]:
train_env = flyte.TaskEnvironment(
    name="training",
    image=ml_image,
    resources=flyte.Resources(
        cpu=("2", "4"),                                # request 2, burst to 4
        memory=("4Gi", "8Gi"),
        disk="20Gi",                                   # ephemeral scratch space
        shm="2Gi",                                     # /dev/shm, e.g. torch DataLoader
        gpu=flyte.GPU(device="T4", quantity=1),        # or A100, L4, H100, ...
    ),
)

# GPU variants your platform team may expose via node pools:
#   flyte.GPU(device="A100", quantity=2)
#   flyte.GPU(device="A100", quantity=1, partition="1g.5gb")   # MIG slice
#   flyte.TPU(device="V5P", partition="2x2x1")                 # GCP TPUs

### At the Kubernetes level

Each resource maps 1:1 to the pod spec: `cpu`/`memory` → container requests/limits,
`gpu` → `nvidia.com/gpu` + node selector for the device type, `shm` → an emptyDir mounted
at `/dev/shm`. Whether a `T4` or `A100` request is schedulable depends on the **node pools**
your platform team provisioned — see [appendix A](./appendix/A-platform-config-map.md) → *Node pools*.

## 5. Multi-environment pipelines

Real pipelines mix heavy and light steps. Put the heavy steps in their own environment and
keep the orchestrating task lightweight — you don't want the coordinator holding a GPU while
it waits. `depends_on` tells the deployment which environments travel together.

In [ ]:
from typing import List

light_env = flyte.TaskEnvironment(
    name="orchestrator",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    depends_on=[train_env],           # orchestrator calls tasks from train_env
)


@train_env.task
async def train_model(seed: int) -> float:
    from sklearn.datasets import make_classification
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score

    X, y = make_classification(n_samples=2_000, random_state=seed)
    score = cross_val_score(RandomForestClassifier(random_state=seed), X, y, cv=3).mean()
    return float(score)


@light_env.task
async def experiment(seeds: List[int]) -> float:
    import asyncio
    scores = await asyncio.gather(*[train_model(seed=s) for s in seeds])
    return max(scores)

> The cell above requests a GPU per training task. If your workshop project has no free GPU
> quota right now, run the CPU variant below — same code path, GPU removed at **call time**.

## 6. `.override()` — call-time configuration

Any task call can override resources, retries, timeout, cache, queue... without touching the
task definition. This is *the* mechanism for "same code, different sizing" — and for error
recovery patterns you'll see in [02](./02-parallelism-and-resilience.ipynb).

In [ ]:
run = await flyte.run.aio(
    experiment.override(resources=flyte.Resources(cpu="1", memory="1Gi")),
    seeds=[1, 2, 3],
)
print(run.url)
await run.wait.aio()
run.outputs()

> Note: overriding `experiment` here only right-sizes the orchestrator; each `train_model`
> call still uses `train_env`'s resources. To downsize those too:
> `await train_model.override(resources=flyte.Resources(cpu='1', memory='2Gi'))(seed=s)`
> inside the orchestrating task.

## Further reading

- Next: [02-parallelism-and-resilience](./02-parallelism-and-resilience.ipynb)
- Union docs → User guide → Task configuration (images, resources, environments)